# 06 Ablations: where does the GAT's gain come from? (C1 Adaptive Curriculum Engine)

The same GAT v2 as notebook 04 (node features, `bkt_p_correct` offset, learner features), trained four ways on the **same 5 folds**:

| Variant | Neighbours the attention looks at | Weights |
|---|---|---|
| **full** | direct prerequisites (notebook 02 graph) | learned attention |
| **no_attention** | direct prerequisites | equal weights (plain mean) |
| **similarity_graph** | the most similar exercises (expert similarity >= 6, notebook 02), at most as many as the prerequisite graph allows | learned attention |
| **no_graph** | none (the exercise itself only) | not applicable |

Each variant is compared with **full** by a paired t-test over the 5 folds. If **full** is better than **no_attention**, attention helps; better than **similarity_graph**, the *prerequisite* structure (not just relatedness) helps; better than **no_graph**, the graph helps at all. Node-level numbers (depth, prerequisite count, mean prerequisite mastery) stay the same in every variant, so only the set of neighbours and how they are weighted changes.

Needs notebooks 01, 02 and 03. Use a **GPU runtime** (T4); about 20 trainings, plus one feature pass.

In [1]:
# Where outputs are stored: "drive", "runtime" or "local" (see notebook 01).
STORAGE = "drive"
# True: 3,000 training students, 1 epoch, 2 folds, for a fast check.
QUICK = False

In [2]:
import sys
!{sys.executable} -m pip install -q pyarrow scikit-learn networkx scipy

from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if not ON_COLAB:
    STORAGE = "local"

if STORAGE == "drive":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = Path("/content/drive/MyDrive/adaptlearn-c1")
    except Exception as exc:
        print(f"Drive could not be mounted ({type(exc).__name__}: {exc}). Using runtime storage.")
        STORAGE = "runtime"
if STORAGE == "runtime":
    BASE = Path("/content/adaptlearn-c1")
elif STORAGE == "local":
    BASE = Path("../data").resolve()

PROCESSED = BASE / "junyi_processed"
OUT_DIR = PROCESSED / "ablations"
OUT_DIR.mkdir(parents=True, exist_ok=True)
for needed in ("interactions.parquet", "bkt/params.csv", "graph/edge_index.npy", "graph/similarity_edge_index.npy"):
    assert (PROCESSED / needed).exists(), f"{needed} missing: run notebooks 01, 02 and 03 first."
print("Storage:", STORAGE, "->", BASE)

Mounted at /content/drive
Storage: drive -> /content/drive/MyDrive/adaptlearn-c1


## 1. Load answers, splits, both graphs and the BKT parameters

In [3]:
import json
import math
import numpy as np
import pandas as pd
import networkx as nx
import torch

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

CONFIG = {"SEED": SEED, "QUICK": QUICK, "HEADS": 4, "HIDDEN": 16, "CLASSIFIER": 32,
          "EPOCHS": 1 if QUICK else 3, "FOLDS": 2 if QUICK else 5, "BATCH": 4096, "LR": 3e-3,
          "WEIGHT_DECAY": 1e-5, "LEAKY_SLOPE": 0.2, "RECENT": 5,
          "VARIANTS": ["full", "no_attention", "similarity_graph", "no_graph"]}

inter = pd.read_parquet(PROCESSED / "interactions.parquet", columns=["user_id", "exercise_idx", "step", "correct"])
inter = inter.sort_values(["user_id", "step"], kind="mergesort").reset_index(drop=True)
splits = json.loads((PROCESSED / "splits.json").read_text())
exercise_index = json.loads((PROCESSED / "exercise_index.json").read_text())
names = sorted(exercise_index, key=exercise_index.get)
n_ex = len(names)
edge_index = np.load(PROCESSED / "graph" / "edge_index.npy")             # row 0 prerequisite, row 1 exercise
sim_index = np.load(PROCESSED / "graph" / "similarity_edge_index.npy")    # both directions
bkt = pd.read_csv(PROCESSED / "bkt" / "params.csv").set_index("exercise").loc[names]

fold_of = {u: int(k) for k, users in splits["folds"].items() for u in users}
if QUICK:
    keep = set(rng.choice(sorted(fold_of), min(3_000, len(fold_of)), replace=False))
    inter = inter[inter.user_id.isin(keep)].reset_index(drop=True)
inter = inter[inter.user_id.isin(fold_of)].reset_index(drop=True)  # cross-validation only: test students stay unseen
print(f"{inter.user_id.nunique():,} training students, {len(inter):,} answers")

G = nx.DiGraph()
G.add_nodes_from(range(n_ex))
G.add_edges_from(zip(edge_index[0].tolist(), edge_index[1].tolist()))
assert nx.is_directed_acyclic_graph(G)
depth = np.zeros(n_ex)
for v in nx.topological_sort(G):
    preds = list(G.predecessors(v))
    depth[v] = 1 + max(depth[p] for p in preds) if preds else 0
prereqs = [sorted(G.predecessors(v)) for v in range(n_ex)]
K = max(len(p) for p in prereqs)

similar = [set() for _ in range(n_ex)]
for a, b in sim_index.T.tolist():
    if a != b:
        similar[a].add(b)
        similar[b].add(a)
# Same neighbour budget as the prerequisite graph, so the comparison is about which neighbours, not how many.
sims = [sorted(s)[:K] for s in similar]
print(f"K = {K}; exercises with prerequisites: {sum(bool(p) for p in prereqs)}, "
      f"with similar exercises: {sum(bool(s) for s in sims)}")

device: cuda
24,000 training students, 4,781,106 answers
K = 7; exercises with prerequisites: 707, with similar exercises: 339


## 2. Features from answers before *t* only

The same features as notebook 04 v2, computed once for both neighbour sets.

In [4]:
from collections import deque
from tqdm.auto import tqdm

FEATURES = ["mastery", "log_evidence", "log_depth", "log_prereq_count", "mean_prereq_mastery", "bkt_p_correct"]
BKT_P = FEATURES.index("bkt_p_correct")
F, L = len(FEATURES), 3
P_INIT, P_LEARN, P_GUESS, P_SLIP = (bkt[c].to_numpy().tolist() for c in ("p_init", "p_learn", "p_guess", "p_slip"))
LOG_DEPTH = np.log1p(depth).tolist()
LOG_PREREQS = [math.log1p(len(p)) for p in prereqs]

def node_features(v, state, count):
    ps = prereqs[v]
    mean_pm = sum(state[p] for p in ps) / len(ps) if ps else 1.0
    m = state[v]
    return (m, math.log1p(count[v]), LOG_DEPTH[v], LOG_PREREQS[v], mean_pm,
            m * (1 - P_SLIP[v]) + (1 - m) * P_GUESS[v])

n = len(inter)
X_t = np.zeros((n, F), np.float32)
X_n, M = np.zeros((n, K, F), np.float32), np.zeros((n, K), bool)    # prerequisite neighbours
X_s, M_s = np.zeros((n, K, F), np.float32), np.zeros((n, K), bool)  # similarity neighbours
X_l = np.zeros((n, L), np.float32)
p_bkt = np.zeros(n, np.float32)

users = inter.user_id.to_numpy()
ex = inter.exercise_idx.to_numpy().tolist()
y = inter.correct.to_numpy().astype(bool)
y_list = y.tolist()
new_student = np.r_[True, users[1:] != users[:-1]].tolist()

for i in tqdm(range(n), mininterval=5):
    if new_student[i]:
        state, count = list(P_INIT), [0] * n_ex
        answered, right, recent, previous = 0, 0, deque(maxlen=CONFIG["RECENT"]), 0.5
    X_l[i] = ((right + 1) / (answered + 2), (sum(recent) + 1) / (len(recent) + 2), previous)
    v = ex[i]
    X_t[i] = node_features(v, state, count)
    for j, u in enumerate(prereqs[v]):
        X_n[i, j] = node_features(u, state, count)
        M[i, j] = True
    for j, u in enumerate(sims[v]):
        X_s[i, j] = node_features(u, state, count)
        M_s[i, j] = True
    m, g, s = state[v], P_GUESS[v], P_SLIP[v]
    p_bkt[i] = m * (1 - s) + (1 - m) * g
    post = m * (1 - s) / (m * (1 - s) + (1 - m) * g) if y_list[i] else m * s / (m * s + (1 - m) * (1 - g))
    state[v] = post + (1 - post) * P_LEARN[v]
    count[v] += 1
    answered += 1
    right += y_list[i]
    recent.append(y_list[i])
    previous = float(y_list[i])

fold = inter.user_id.map(fold_of).astype(int).to_numpy()
print(f"answers with prerequisites: {M.any(1).mean():.1%}, with similar exercises: {M_s.any(1).mean():.1%}")

  0%|          | 0/4781106 [00:00<?, ?it/s]

answers with prerequisites: 88.0%, with similar exercises: 81.9%


## 3. One model, four variants

In [5]:
import torch.nn as nn
import torch.nn.functional as fn


class AblationGAT(nn.Module):
    """GAT v2 of notebook 04; `attention=False` gives every present neighbour (and self) equal weight."""

    def __init__(self, f, l, heads, hidden, classifier, slope, attention=True):
        super().__init__()
        self.heads, self.hidden, self.slope, self.attention = heads, hidden, slope, attention
        self.W = nn.Linear(f, heads * hidden, bias=False)
        self.a_dst = nn.Parameter(torch.empty(heads, hidden))
        self.a_src = nn.Parameter(torch.empty(heads, hidden))
        nn.init.xavier_uniform_(self.a_dst)
        nn.init.xavier_uniform_(self.a_src)
        self.fc1 = nn.Linear(heads * hidden + f + l, classifier)
        self.fc2 = nn.Linear(classifier, 1)
        nn.init.zeros_(self.fc2.weight)  # start exactly at BKT
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x_t, x_n, mask, x_l):
        B, k, _ = x_n.shape
        H, D = self.heads, self.hidden
        h = self.W(torch.cat([x_t.unsqueeze(1), x_n], 1)).view(B, 1 + k, H, D)
        present = torch.cat([torch.ones(B, 1, dtype=torch.bool, device=mask.device), mask], 1)
        if self.attention:
            score = (h[:, :1] * self.a_dst).sum(-1) + (h * self.a_src).sum(-1)
            score = fn.leaky_relu(score, self.slope).masked_fill(~present.unsqueeze(-1), float("-inf"))
            alpha = torch.softmax(score, dim=1)
        else:
            alpha = (present.float() / present.sum(1, keepdim=True)).unsqueeze(-1).expand(-1, -1, H)
        z = fn.elu((alpha.unsqueeze(-1) * h).sum(1)).reshape(B, H * D)
        base = torch.logit(x_t[:, BKT_P].clamp(1e-4, 1 - 1e-4))
        return base + self.fc2(torch.relu(self.fc1(torch.cat([z, x_t, x_l], -1)))).squeeze(-1)


T_xt, T_xl = torch.from_numpy(X_t).to(DEVICE), torch.from_numpy(X_l).to(DEVICE)
T_y = torch.from_numpy(y.astype(np.float32)).to(DEVICE)
NEIGHBOURS = {
    "full": (torch.from_numpy(X_n).to(DEVICE), torch.from_numpy(M).to(DEVICE)),
    "similarity_graph": (torch.from_numpy(X_s).to(DEVICE), torch.from_numpy(M_s).to(DEVICE)),
}
NEIGHBOURS["no_attention"] = NEIGHBOURS["full"]
NEIGHBOURS["no_graph"] = (NEIGHBOURS["full"][0], torch.zeros_like(NEIGHBOURS["full"][1]))


def new_model(variant):
    return AblationGAT(F, L, CONFIG["HEADS"], CONFIG["HIDDEN"], CONFIG["CLASSIFIER"], CONFIG["LEAKY_SLOPE"],
                       attention=variant != "no_attention").to(DEVICE)


def train(variant, rows, seed=SEED):
    torch.manual_seed(seed)
    model = new_model(variant)
    xn, mk = NEIGHBOURS[variant]
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"])
    rows = torch.from_numpy(rows).to(DEVICE)
    g = torch.Generator(device="cpu").manual_seed(seed)
    for _ in range(CONFIG["EPOCHS"]):
        model.train()
        order = rows[torch.randperm(len(rows), generator=g).to(DEVICE)]
        for b in range(0, len(order), CONFIG["BATCH"]):
            idx = order[b:b + CONFIG["BATCH"]]
            loss = fn.binary_cross_entropy_with_logits(model(T_xt[idx], xn[idx], mk[idx], T_xl[idx]), T_y[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
    return model


@torch.no_grad()
def predict(model, variant, rows):
    model.eval()
    xn, mk = NEIGHBOURS[variant]
    out = []
    for b in range(0, len(rows), 65536):
        idx = torch.from_numpy(rows[b:b + 65536]).to(DEVICE)
        out.append(torch.sigmoid(model(T_xt[idx], xn[idx], mk[idx], T_xl[idx])).cpu().numpy())
    return np.concatenate(out)

## 4. Cross-validation: every variant on the same folds

In [6]:
import time
from sklearn.metrics import roc_auc_score
from scipy.stats import ttest_rel

rows_cv = []
for k in range(CONFIG["FOLDS"]):
    train_rows, val_rows = np.flatnonzero(fold != k), np.flatnonzero(fold == k)
    result = {"fold": k, "bkt_only": roc_auc_score(y[val_rows], p_bkt[val_rows])}
    for variant in CONFIG["VARIANTS"]:
        start = time.time()
        model = train(variant, train_rows)
        result[variant] = roc_auc_score(y[val_rows], predict(model, variant, val_rows))
        print(f"fold {k} {variant:<17} AUC {result[variant]:.4f}  ({(time.time() - start) / 60:.1f} min)")
    rows_cv.append(result)

cv = pd.DataFrame(rows_cv)
summary = []
for variant in ["bkt_only"] + CONFIG["VARIANTS"]:
    line = {"variant": variant, "auc_mean": cv[variant].mean(), "auc_sd": cv[variant].std()}
    if variant != "full":
        t, p = ttest_rel(cv["full"], cv[variant])
        line |= {"full_minus_variant": (cv["full"] - cv[variant]).mean(), "paired_t": t, "paired_p": p}
    summary.append(line)
summary = pd.DataFrame(summary)
display(cv.round(4))
summary.round(4)

fold 0 full              AUC 0.7959  (0.2 min)
fold 0 no_attention      AUC 0.7957  (0.1 min)
fold 0 similarity_graph  AUC 0.7959  (0.1 min)
fold 0 no_graph          AUC 0.7955  (0.1 min)
fold 1 full              AUC 0.7991  (0.1 min)
fold 1 no_attention      AUC 0.7988  (0.1 min)
fold 1 similarity_graph  AUC 0.7991  (0.1 min)
fold 1 no_graph          AUC 0.7981  (0.1 min)
fold 2 full              AUC 0.7919  (0.1 min)
fold 2 no_attention      AUC 0.7919  (0.1 min)
fold 2 similarity_graph  AUC 0.7919  (0.1 min)
fold 2 no_graph          AUC 0.7913  (0.1 min)
fold 3 full              AUC 0.7945  (0.1 min)
fold 3 no_attention      AUC 0.7945  (0.1 min)
fold 3 similarity_graph  AUC 0.7946  (0.1 min)
fold 3 no_graph          AUC 0.7937  (0.1 min)
fold 4 full              AUC 0.7970  (0.1 min)
fold 4 no_attention      AUC 0.7969  (0.1 min)
fold 4 similarity_graph  AUC 0.7972  (0.1 min)
fold 4 no_graph          AUC 0.7966  (0.1 min)


,fold,bkt_only,full,no_attention,similarity_graph,no_graph
0,0,0.7854,0.7959,0.7957,0.7959,0.7955
1,1,0.7882,0.7991,0.7988,0.7991,0.7981
2,2,0.7817,0.7919,0.7919,0.7919,0.7913
3,3,0.7813,0.7945,0.7945,0.7946,0.7937
4,4,0.7858,0.7970,0.7969,0.7972,0.7966


,variant,auc_mean,auc_sd,full_minus_variant,paired_t,paired_p
0,bkt_only,0.7845,0.0029,0.0112,21.6275,0.0000
1,full,0.7957,0.0027,NaN,NaN,NaN
2,no_attention,0.7956,0.0026,0.0001,1.9731,0.1197
3,similarity_graph,0.7957,0.0027,-0.0000,-0.8361,0.4501
4,no_graph,0.7950,0.0027,0.0006,5.8086,0.0044


## 5. Save results and a manifest

In [7]:
import hashlib, platform, sklearn
from datetime import datetime, timezone

cv.to_csv(OUT_DIR / "cv_folds.csv", index=False)
summary.to_csv(OUT_DIR / "summary.csv", index=False)
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "torch": torch.__version__, "scikit-learn": sklearn.__version__, "networkx": nx.__version__}
manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "config": CONFIG | {"K": int(K)},
    "versions": VERSIONS,
    "sha256": {f.name: hashlib.sha256(f.read_bytes()).hexdigest()
               for f in sorted(OUT_DIR.glob("*")) if f.is_file() and f.name != "manifest.json"},
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

{
  "created_at": "2026-10-10T14:14:31.023975+00:00",
  "config": {
    "SEED": 42,
    "QUICK": false,
    "HEADS": 4,
    "HIDDEN": 16,
    "CLASSIFIER": 32,
    "EPOCHS": 3,
    "FOLDS": 5,
    "BATCH": 4096,
    "LR": 0.003,
    "WEIGHT_DECAY": 1e-05,
    "LEAKY_SLOPE": 0.2,
    "RECENT": 5,
    "VARIANTS": [
      "full",
      "no_attention",
      "similarity_graph",
      "no_graph"
    ],
    "K": 7
  },
  "versions": {
    "python": "3.13.15",
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "torch": "2.11.0+cu130",
    "scikit-learn": "1.6.1",
    "networkx": "3.7"
  },
  "sha256": {
    "cv_folds.csv": "04d0159c56692a5090da12f048553d793cde3b117caacd2cff8334e4fb1cd28d",
    "summary.csv": "af9190968336cdb668939408cd6efe580c8e842a15368da7bdd1569744a1e3a0"
  }
}


In [8]:
if STORAGE == "runtime":
    import shutil
    print("Download before stopping the runtime:", shutil.make_archive("/content/ablation_outputs", "zip", OUT_DIR))
else:
    print("Outputs stored in", OUT_DIR)

Outputs stored in /content/drive/MyDrive/adaptlearn-c1/junyi_processed/ablations


## For the report

Report the summary table: mean and SD of AUC per variant over the 5 folds, and for each variant the mean difference from **full** with its paired t-test. Read it as:

- full > no_attention: the learned attention adds to a plain average of prerequisites;
- full > similarity_graph: prerequisite edges carry information that mere similarity does not;
- full > no_graph: the graph adds to the learner and BKT features at all.

A small or non-significant difference is a finding too; report it as it is. Only training students are used (the held-out test students stay unseen until the final results notebook).

Before committing: **Clear all outputs**.